Here's the master notebooks for this project, connecting a bunch of separate python scripts.
If this project was to be continued to run as a scheduled job, I'd have built this as a script as well.

In [13]:
#We have a BUNCH of modules, so let's make sure we reload them automatically when we edit them.
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../src")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Time for imports. This is a bit leaner that would be expected due to the modular project of this project.
We also build in some toggles, since this is the master interface.

When running this for the first time, set RESET_DB and SCRAPE to True.

In order to change the list of cities, just edit the list below.

In [14]:
import pandas as pd
from db import get_engine, insert_ignore
from schema import create_tables, reset_database
from scrape import cityscraper, popscraper
from weather import load_cities, get_weather_data, weather_to_df, get_weather_forecast, forecast_to_df
from airports import get_airports, get_arrivals, flights_to_df

RESET_DB = False     # True wipes everything
SCRAPE   = True   # True re-scrapes Wikipedia (only needed for new cities)
FLIGHTS  = True   # True fetches tomorrow's arrivals (costs API quota)

euro_cities = ["Berlin", "Hamburg", "Munich", "Cologne", "Frankfurt", "London",
               "Kiruna", "Stockholm", "Prague", "Paris", "Kiev", "Madrid"]

In [15]:
#And here is where we execute the reset. It is still locked behind the safety above.

if RESET_DB:
    reset_database()
create_tables()
engine = get_engine()

In [16]:
#Here's the cell for running the webscraping. It is also locked behind the safety.
if SCRAPE:
    cities_df = cityscraper(euro_cities) #scrapes city country and coords from wikipedia into a dataframe
    pops_df = popscraper(euro_cities) #scrapes city population from wikipedia into a dataframe

    cities_df[["city_name"]].to_sql("cities", con=engine, if_exists="append",
                                    index=False, method=insert_ignore)
    keys = pd.read_sql("cities", con=engine)
    #the above injects the results into the database, and then reads back the city_id primary keys for the next step.

    cities_df.merge(keys, on="city_name")[["city_id", "longitude", "latitude"]] \
        .to_sql("cities_coords", con=engine, if_exists="append", index=False, method=insert_ignore)
    pops_df.merge(keys, on="city_name")[["city_id", "population", "year_retrieved"]] \
        .to_sql("city_pop", con=engine, if_exists="append", index=False, method=insert_ignore)
    #the above populates two more tables, using the key we just pulled
    airports_df = get_airports(load_cities(engine))
    display(airports_df)   # check for unwanted airfields before inserting
    airports_df.to_sql("airports", con=engine, if_exists="append", index=False, method=insert_ignore)
    #and getting the airport data

,icao,iata,airport_name,city_id
0,EDDT,TXL,Berlin -Tegel,1
1,EDDB,BER,Berlin Brandenburg,1
2,EDDK,CGN,Cologne Bonn,4
3,EDDL,DUS,Düsseldorf,4
4,EDDF,FRA,Frankfurt-am-Main,5
5,EDDH,HAM,Hamburg,2
6,UKKK,IEV,Kyiv Zhuliany,11
7,UKBB,KBP,Kyiv Boryspil,11
8,ESNQ,KRN,,7
9,EGLC,LCY,London City,6


In [17]:
#Pulling and injecting weather data with an api from openweathermap.org. This is not locked behind a safety, because it is safe to run multiple times.
cities = load_cities(engine)

weather_df = weather_to_df(get_weather_data(cities))
forecasts_df = forecast_to_df(get_weather_forecast(cities))

n1 = weather_df.to_sql("weather_actual", con=engine, if_exists="append", index=False, method=insert_ignore)
n2 = forecasts_df.to_sql("weather_forecast", con=engine, if_exists="append", index=False, method=insert_ignore)
print(f"weather_actual: {n1} new rows, weather_forecast: {n2} new rows")

weather_actual: 3 new rows, weather_forecast: 0 new rows


In [18]:
#and finally we can pull tomorrows arrivals.
if FLIGHTS:
    icaos = pd.read_sql("airports", con=engine)["icao"].tolist()
    flights_df = flights_to_df(get_arrivals(icaos))
    n3 = flights_df.to_sql("flights", con=engine, if_exists="append", index=False, method=insert_ignore)
    print(f"flights: {n3} new rows")

flights: 5742 new rows


In [19]:
# Test cell.
pd.read_sql("""
    SELECT c.city_name, COUNT(*) AS arrivals
    FROM flights f
    JOIN airports a ON f.arrival_icao = a.icao
    JOIN cities c   ON a.city_id = c.city_id
    GROUP BY c.city_name
    ORDER BY arrivals DESC
""", con=engine)

,city_name,arrivals
0,London,1561
1,Paris,990
2,Frankfurt,645
3,Madrid,630
4,Munich,497
5,Cologne,381
6,Stockholm,333
7,Berlin,305
8,Prague,216
9,Hamburg,181
